In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def read_inspect_b_data(filename):
    """
    Read data from inspect_b_output.dat file.
    
    Format:
    - Line 1: time (single number)
    - Following lines for each variable (psi, chi, b):
      - mode_index, radial_data1, radial_data2, ...
    - Pattern repeats for each time step
    
    Returns:
    - times: array of simulation times
    - data: list of dictionaries, each containing data for psi, chi, and b
    """
    times = []
    all_data = []
    
    with open(filename, 'r') as f:
        lines = f.readlines()
    
    i = 0
    nrad = None  # Will be determined from first data line
    
    while i < len(lines):
        # Initialize data storage for this time step
        timestep_data = {
            'psi': {'mode_indices': [], 'data': []},
            'chi': {'mode_indices': [], 'data': []},
            'b': {'mode_indices': [], 'data': []},
            'nrad': nrad
        }
        
        # Read data for all three variables (psi, chi, b) - each has 2 lines: time + data
        variable_names = ['psi', 'chi', 'b']
        time_for_step = None
        
        for var_idx, var_name in enumerate(variable_names):
            # Check if we have enough lines remaining
            if i + 1 >= len(lines):
                break
                
            # Read time line (single number)
            time_line = lines[i].strip()
            try:
                time = float(time_line)
                if time_for_step is None:
                    time_for_step = time  # Store the time for this time step
                i += 1
            except ValueError:
                print(f"Warning: Could not parse time from line {i}: {time_line}")
                i += 1
                continue
            
            # Read data line (mode_index + radial_data)
            if i >= len(lines):
                break
                
            data_line = lines[i].strip()
            
            # Try to parse as data line (mode_index + radial_data)
            try:
                data_values = []
                for x in data_line.split():
                    try:
                        data_values.append(float(x.rstrip(',')))
                    except ValueError:
                        print(f"Warning: Could not convert '{x}' to float, setting to 0.0")
                        data_values.append(0.0)
                
                if len(data_values) < 2:
                    print(f"Warning: Insufficient data in line {i} for {var_name}")
                    i += 1
                    continue
                
                # Determine nrad from first successful data line
                if nrad is None:
                    nrad = len(data_values) - 1  # First number is mode index
                    timestep_data['nrad'] = nrad
                
                # Check if this has the expected number of data points
                if len(data_values) == nrad + 1:  # mode_index + nrad data points
                    mode_index = int(data_values[0])
                    radial_data = np.array(data_values[1:])
                    
                    timestep_data[var_name]['mode_indices'] = np.array([mode_index])
                    timestep_data[var_name]['data'] = np.array([radial_data])  # Shape: (1, n_radial)
                    i += 1
                else:
                    print(f"Warning: Unexpected data size in line {i} for {var_name}: expected {nrad+1}, got {len(data_values)}")
                    i += 1
                    
            except ValueError:
                print(f"Warning: Could not parse {var_name} data from line {i}: {data_line}")
                i += 1
                continue
        
        # Store the complete time step data if we got a valid time and at least some data
        if time_for_step is not None and any(len(timestep_data[var]['data']) > 0 for var in variable_names):
            times.append(time_for_step)
            timestep_data['nrad'] = nrad
            all_data.append(timestep_data)
    
    return np.array(times), all_data

def plot_data(times, all_data, variable='psi', mode_to_plot=1, radial_point=1, ax=None):
    """
    Plot time evolution of a specific radial point for a specific mode and variable.
    
    Parameters:
    - times: array of simulation times
    - all_data: list of data dictionaries
    - variable: variable to plot ('psi', 'chi', or 'b')
    - mode_to_plot: azimuthal mode index to plot
    - radial_point: radial grid point index to plot (1-based)
    - ax: optional matplotlib axes to plot on (if None, creates new figure)
    
    Returns:
    - ax: matplotlib axes object (if data was plotted)
    """
    if variable not in ['psi', 'chi', 'b']:
        print(f"Error: variable must be one of 'psi', 'chi', 'b', got '{variable}'")
        return None
        
    values = []
    valid_times = []
    
    for i, (time, data_dict) in enumerate(zip(times, all_data)):
        if variable not in data_dict or len(data_dict[variable]['data']) == 0:
            continue
            
        mode_indices = data_dict[variable]['mode_indices']
        data = data_dict[variable]['data']
        
        # Find the requested mode
        mode_mask = mode_indices == mode_to_plot
        if np.any(mode_mask):
            mode_data = data[mode_mask][0]  # Take first match
            if 0 <= radial_point-1 < len(mode_data):
                values.append(mode_data[radial_point-1])  # Convert to 0-based
                valid_times.append(time)
    
    if values:
        created_new_figure = False
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 6))
            created_new_figure = True
        
        # Add label to distinguish different lines
        ax.plot(valid_times, values, linewidth=2, 
               label=f'{variable.upper()}: Mode {mode_to_plot}, Radial {radial_point}')
        ax.set_xlabel('Simulation Time')
        ax.set_ylabel('Value')
        ax.set_title(f'Time Evolution Comparison')
        ax.grid(True, alpha=0.3)
        ax.legend()  # Add legend to show different lines
        
        if created_new_figure:
            plt.show()
        
        return ax
    else:
        print(f"No data found for {variable} mode {mode_to_plot}")
        return None

def plot_spectrum(times, all_data, variable='psi', mode_to_plot=1, time_indices=None):
    """
    Plot the spectrum of a specific mode as a function of radial points.
    
    Parameters:
    - times: array of simulation times
    - all_data: list of data dictionaries
    - variable: variable to plot ('psi', 'chi', or 'b')
    - mode_to_plot: azimuthal mode index to plot
    - time_indices: optional list of time indices to plot (if None, plots all times)
    """
    if variable not in ['psi', 'chi', 'b']:
        print(f"Error: variable must be one of 'psi', 'chi', 'b', got '{variable}'")
        return
        
    if time_indices is None:
        time_indices = range(len(times))
    
    plt.figure(figsize=(14, 10))
    
    # Define colors for different time steps - use a colormap that can handle many colors
    if len(time_indices) <= 10:
        colors = plt.cm.tab10(np.linspace(0, 1, 10))[:len(time_indices)]
    else:
        # Use a continuous colormap for many time steps
        colors = plt.cm.hsv(np.linspace(0, 1, len(time_indices)))
    
    for i, time_idx in enumerate(time_indices):
        if time_idx >= len(all_data):
            print(f"Warning: time index {time_idx} out of range")
            continue
            
        data_dict = all_data[time_idx]
        
        if variable not in data_dict or len(data_dict[variable]['data']) == 0:
            continue
            
        mode_indices = data_dict[variable]['mode_indices']
        data = data_dict[variable]['data']
        
        # Find the requested mode
        mode_mask = mode_indices == mode_to_plot
        if np.any(mode_mask):
            mode_data = data[mode_mask][0]  # Take first match
            spectrum = np.abs(mode_data)  # Magnitude for y-axis
            signs = np.sign(mode_data)    # Sign for color coding
            
            # Create radial grid (assuming uniform spacing)
            radial_points = np.arange(len(spectrum))+1
            
            # Check if modes (except first) share the same sign (allow ≤5 outliers)
            signs_tail = signs[1:]  # Exclude first mode (gauge freedom)
            n_positive = np.sum(signs_tail > 0)
            n_negative = np.sum(signs_tail < 0)
            n_total = len(signs_tail)

            # Dominant sign if ≤20 outliers
            mostly_positive = n_negative <= 2
            mostly_negative = n_positive <= 2
            
            color = colors[i]
            
            # Plot as line if most tail modes share same sign, mark outliers with circles
            if mostly_positive or mostly_negative:
                dominant_sign = 1 if mostly_positive else -1
                linestyle = '-' if mostly_positive else '--'
                
                # Plot main line for all modes matching dominant sign
                conforming_mask = signs * dominant_sign > 0
                plt.plot(radial_points[conforming_mask], spectrum[conforming_mask], 
                        linestyle=linestyle, color=color, linewidth=1.5, alpha=0.7, 
                        label=f't = {times[time_idx]:.3f}')
                
                # Mark outliers (non-conforming modes) with circles
                outlier_mask = (signs * dominant_sign < 0) & (signs != 0)
                if np.any(outlier_mask):
                    plt.plot(radial_points[outlier_mask], spectrum[outlier_mask], 
                            'o', color=color, markersize=6, alpha=0.9, 
                            markerfacecolor='none', markeredgewidth=2)
            else:
                # Use markers for mixed signs
                positive_mask = signs >= 0
                negative_mask = signs < 0
                
                if np.any(positive_mask):
                    plt.plot(radial_points[positive_mask], spectrum[positive_mask], 
                            '|', color=color, markersize=3, alpha=0.7,
                            label=f't = {times[time_idx]:.3f}' if i == 0 else "")
                
                if np.any(negative_mask):
                    plt.plot(radial_points[negative_mask], spectrum[negative_mask], 
                            '_', color=color, markersize=3, alpha=0.7,
                            label="" if i > 0 else "")
    
    # Add vertical line at 2/3 of total radial points
    if time_indices and len(time_indices) > 0:
        # Get the number of radial points from the first valid data
        first_valid_idx = None
        for time_idx in time_indices:
            if time_idx < len(all_data):
                data_dict = all_data[time_idx]
                if variable in data_dict and len(data_dict[variable]['data']) > 0:
                    first_valid_idx = time_idx
                    break
        
        if first_valid_idx is not None:
            nrad = len(all_data[first_valid_idx][variable]['data'][0])
            two_thirds_point = (2/3) * nrad
            plt.axvline(x=two_thirds_point, color='red', linestyle='--', linewidth=2, 
                       alpha=0.8, label=f'2/3 radial points ({two_thirds_point:.1f})')

    plt.xlabel('Radial Mode Index')
    plt.ylabel('|Amplitude| (Log Scale)')
    plt.title(f'Radial Spectrum of {variable.upper()} Mode {mode_to_plot}\n(Solid: all +, Dashed: all -, Circles: outliers, Markers: mixed)')
    plt.grid(True, alpha=0.3)
    
    # Add custom legend entries for marker indication
    from matplotlib.lines import Line2D
    custom_lines = [Line2D([0], [0], color='gray', marker='o', linestyle='', markersize=8),
                    Line2D([0], [0], color='gray', marker='s', linestyle='', markersize=8)]
    plt.legend(custom_lines, ['Positive amplitude', 'Negative amplitude'], 
              loc='lower left')
    
    plt.xscale('log')
    plt.yscale('log')  # Often useful for spectrum data
    plt.show()

def print_data_summary(times, all_data):
    """Print summary of the loaded data."""
    print(f"Total time steps: {len(times)}")
    if len(times) > 0:
        print(f"Time range: {times[0]:.6f} to {times[-1]:.6f}")
    
    if all_data:
        first_data = all_data[0]
        print(f"Number of radial points: {first_data['nrad']}")
        
        # Print information about each variable
        variables = ['psi', 'chi', 'b']
        for var in variables:
            if var in first_data and len(first_data[var]['data']) > 0:
                modes = first_data[var]['mode_indices']
                print(f"Available modes for {var.upper()}: {modes}")
            else:
                print(f"No data found for {var.upper()}")
        
        # Check consistency across time steps
        nrad_values = [data['nrad'] for data in all_data]
        if len(set(nrad_values)) > 1:
            print(f"Warning: Inconsistent number of radial points: {set(nrad_values)}")
        
        # Check data completeness
        print("\nData completeness check:")
        for var in variables:
            time_steps_with_data = sum(1 for data in all_data if var in data and len(data[var]['data']) > 0)
            print(f"  {var.upper()}: {time_steps_with_data}/{len(all_data)} time steps have data")


In [ ]:
def read_hyperv_data(filename):
    """
    Read hyperviscosity/hyperdiffusivity data from hyperv.dat file.
    
    Format for each adjustment step:
    - 6 rows total per time step
    - First 3 rows: hyperviscosity (hyperv_r, hyperv_t, hyperv_x)
    - Next 3 rows: hyperdiffusivity (kappa_r, kappa_t, kappa_x)
    - Each row: time, nu_value, mode_data1, mode_data2, ...
    
    Returns:
    - times: array of adjustment times
    - hyperv_data: dict with keys 'r', 't', 'x', each containing time series of mode data
    - kappa_data: dict with keys 'r', 't', 'x', each containing time series of mode data
    - nu_values: dict with time series of nu values for each direction
    """
    times = []
    hyperv_data = {'r': [], 't': [], 'x': []}
    kappa_data = {'r': [], 't': [], 'x': []}
    nu_values = {'hyperv': {'r': [], 't': [], 'x': []}, 
                 'kappa': {'r': [], 't': [], 'x': []}}
    
    directions = ['r', 't', 'x']
    
    with open(filename, 'r') as f:
        lines = f.readlines()
    
    i = 0
    while i < len(lines):
        # Check if we have enough lines for a complete time step (6 rows)
        if i + 5 >= len(lines):
            break
            
        current_time = None
        
        # Read hyperviscosity data (first 3 rows)
        for dir_idx, direction in enumerate(directions):
            line = lines[i].strip()
            if not line:
                i += 1
                continue
                
            try:
                # Parse comma-separated values
                values = [float(x.rstrip(',')) for x in line.split()]
                
                if len(values) < 2:
                    print(f"Warning: Insufficient data in line {i}")
                    i += 1
                    continue
                
                time_val = values[0]
                nu_val = values[1]
                mode_data = np.array(values[2:]) if len(values) > 2 else np.array([])
                
                # Store time from first row
                if current_time is None:
                    current_time = time_val
                
                hyperv_data[direction].append(mode_data)
                nu_values['hyperv'][direction].append(nu_val)
                
            except (ValueError, IndexError) as e:
                print(f"Warning: Could not parse hyperv line {i}: {line}")
            
            i += 1
        
        # Read hyperdiffusivity data (next 3 rows)
        for dir_idx, direction in enumerate(directions):
            if i >= len(lines):
                break
                
            line = lines[i].strip()
            if not line:
                i += 1
                continue
                
            try:
                # Parse comma-separated values
                values = [float(x.rstrip(',')) for x in line.split()]
                
                if len(values) < 2:
                    print(f"Warning: Insufficient data in line {i}")
                    i += 1
                    continue
                
                time_val = values[0]
                nu_val = values[1]
                mode_data = np.array(values[2:]) if len(values) > 2 else np.array([])
                
                kappa_data[direction].append(mode_data)
                nu_values['kappa'][direction].append(nu_val)
                
            except (ValueError, IndexError) as e:
                print(f"Warning: Could not parse kappa line {i}: {line}")
            
            i += 1
        
        # Store the time for this complete time step
        if current_time is not None:
            times.append(current_time)
    
    # Convert lists to numpy arrays
    times = np.array(times)
    for direction in directions:
        if hyperv_data[direction]:
            hyperv_data[direction] = np.array(hyperv_data[direction])
        if kappa_data[direction]:
            kappa_data[direction] = np.array(kappa_data[direction])
        for data_type in ['hyperv', 'kappa']:
            if nu_values[data_type][direction]:
                nu_values[data_type][direction] = np.array(nu_values[data_type][direction])
    
    return times, hyperv_data, kappa_data, nu_values

def plot_hyperv_complete(times, hyperv_data, kappa_data, nu_values, 
                        data_type='hyperv', time_indices=None, figsize=(15, 10)):
    """
    Plot comprehensive 2x3 figure: coefficient evolution (top) + mode time-lapse (bottom).
    
    Parameters:
    - times: array of adjustment times
    - hyperv_data: hyperviscosity data dict
    - kappa_data: hyperdiffusivity data dict  
    - nu_values: nu values dict
    - data_type: 'hyperv' or 'kappa' to choose which data to plot
    - time_indices: optional list of time indices for time-lapse plots
    - figsize: figure size tuple
    """
    if time_indices is None:
        # Default: plot every 10th time step for time-lapse
        if len(times) > 20:
            time_indices = list(range(0, len(times), max(1, len(times)//10)))
        else:
            time_indices = list(range(len(times)))
    
    # Select data based on type
    if data_type == 'hyperv':
        mode_data = hyperv_data
        coeff_data = nu_values['hyperv']
        title_prefix = 'Hyperviscosity'
        color_coeff = 'black'
    else:
        mode_data = kappa_data
        coeff_data = nu_values['kappa']
        title_prefix = 'Hyperdiffusivity'
        color_coeff = 'black'
    
    directions = ['r', 't', 'x']
    direction_names = ['Radial (R)', 'Azimuthal (T)', 'Axial (X)']
    
    # Create 2x3 subplot layout
    fig, axes = plt.subplots(2, 3, figsize=figsize)
    fig.suptitle(f'{title_prefix} Analysis: Coefficient Evolution & Mode Time-lapse', fontsize=16)
    
    # Top row: Coefficient evolution over time
    for col, (direction, dir_name) in enumerate(zip(directions, direction_names)):
        ax = axes[0, col]
        
        if len(coeff_data[direction]) > 0:
            ax.plot(times[:len(coeff_data[direction])], 
                   coeff_data[direction], 
                   color=color_coeff, linewidth=2, label=f'{title_prefix} Coefficient')
            
            ax.set_xlabel('Time')
            ax.set_ylabel('Coefficient Value')
            ax.set_title(f'{title_prefix} Coefficient {dir_name}')
            ax.set_yscale('log')
            ax.grid(True, alpha=0.3)
            ax.legend()
        else:
            ax.text(0.5, 0.5, f'No {direction} coefficient data', 
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{title_prefix} Coefficient {dir_name}')
    
    # Bottom row: Mode time-lapse
    # Color scheme: first time step in red, rest in grayscale from light to dark
    colors = []
    if len(time_indices) > 0:
        colors.append('red')  # First time step in red
        if len(time_indices) > 1:
            # Remaining time steps in grayscale from light (0.8) to dark (0.2)
            gray_values = np.linspace(0.8, 0.2, len(time_indices) - 1)
            colors.extend([(gray, gray, gray, 1.0) for gray in gray_values])
    
    for col, (direction, dir_name) in enumerate(zip(directions, direction_names)):
        ax = axes[1, col]
        
        if len(mode_data[direction]) == 0:
            ax.text(0.5, 0.5, f'No {direction} mode data', 
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{title_prefix} Modes {dir_name}')
            continue
        
        # Plot multiple time steps
        for i, time_idx in enumerate(time_indices):
            if time_idx >= len(times) or time_idx >= len(mode_data[direction]):
                continue
                
            mode_spectrum = mode_data[direction][time_idx]
            if len(mode_spectrum) > 0:
                mode_indices = np.arange(len(mode_spectrum)) + 1
                ax.plot(mode_indices, np.abs(mode_spectrum), 
                       color=colors[i], alpha=0.7, linewidth=1.5,
                       label=f't={times[time_idx]:.1f}' if i < 5 else "")
        
        ax.set_xlabel('Mode Index')
        ax.set_ylabel(f'|{title_prefix} Mode|')
        ax.set_title(f'{title_prefix} Modes {dir_name}')
        ax.set_yscale('log')
        ax.grid(True, alpha=0.3)
        if col == 0:  # Only show legend on first subplot
            ax.legend(fontsize=8)
    
    plt.tight_layout()
    plt.show()

def print_hyperv_summary(times, hyperv_data, kappa_data, nu_values):
    """Print summary of the hyperviscosity data."""
    print(f"Total adjustment time steps: {len(times)}")
    if len(times) > 0:
        print(f"Time range: {times[0]:.3f} to {times[-1]:.3f}")
    
    directions = ['r', 't', 'x']
    
    print("\nData shape summary:")
    for direction in directions:
        if len(hyperv_data[direction]) > 0:
            print(f"  Hyperviscosity {direction.upper()}: {hyperv_data[direction].shape}")
        if len(kappa_data[direction]) > 0:
            print(f"  Hyperdiffusivity {direction.upper()}: {kappa_data[direction].shape}")
    
    print("\nCoefficient ranges:")
    for data_type in ['hyperv', 'kappa']:
        type_name = 'Hyperviscosity' if data_type == 'hyperv' else 'Hyperdiffusivity'
        for direction in directions:
            values = nu_values[data_type][direction]
            if len(values) > 0:
                print(f"  {type_name} {direction.upper()}: {values.min():.2e} to {values.max():.2e}")

def process_hyperv(filename='../../output/hyperv.dat'):
    # Read and analyze hyperviscosity data
    try:
        times_hv, hyperv_data, kappa_data, nu_values = read_hyperv_data(filename)

        # Print summary
        print_hyperv_summary(times_hv, hyperv_data, kappa_data, nu_values)
        
        # Select time indices for plotting (every 10th time step or so)
        time_indices = list(range(0, len(times_hv), max(1, len(times_hv)//15)))
        
        # Plot comprehensive analysis for hyperviscosity (velocity)
        plot_hyperv_complete(times_hv, hyperv_data, kappa_data, nu_values, 
                            data_type='hyperv', time_indices=time_indices)
        
        # Plot comprehensive analysis for hyperdiffusivity (density)
        plot_hyperv_complete(times_hv, hyperv_data, kappa_data, nu_values, 
                            data_type='kappa', time_indices=time_indices)
        
    except FileNotFoundError:
        print("hyperv.dat file not found. Make sure the simulation has run with hyperviscosity adjustment.")
    except Exception as e:
        print(f"Error reading hyperviscosity data: {e}")

    return None

In [ ]:
# Read data (automatically handles psi, chi, b)
times, all_data = read_inspect_b_data('../../output/inspect.output')

# # Plot specific variables
# plot_data(times, all_data, variable='psi')   # Plot psi
# plot_data(times, all_data, variable='chi')   # Plot chi  
# plot_data(times, all_data, variable='b')     # Plot b

# Plot spectrum for specific variables
time_to_plot = list(range(0, len(times), 1))
# time_to_plot = list(range(100, 150, 10))
# time_to_plot = list(range(len(times)-1000, len(times), 200))
# time_to_plot = list(range(200, 250))
plot_spectrum(times, all_data, variable='psi', time_indices=time_to_plot)
plot_spectrum(times, all_data, variable='chi', time_indices=time_to_plot)
plot_spectrum(times, all_data, variable='b', time_indices=time_to_plot)

In [ ]:
process_hyperv()

### Functions

In [ ]:
def calculate_curv_from_simulation(filename='../../output/especData.dat', 
                                   spectrum_type='KE', ratio_old=1.0/3.0, 
                                   ratio_new=1.0/4.0, peak_ratio=1.0/3.0, verbose=False):
    """
    Calculate SPECURV curvature values from actual simulation energy spectra.
    Computes both OLD logic (ratio=1/3) and NEW logic (ratio=1/4 with truncation).
    
    IMPORTANT: 
    - Use KE spectrum for hyperviscosity (nu) calculations
    - Use PE spectrum for hyperdiffusivity (kappa) calculations
    
    Parameters:
    -----------
    filename : str
        Path to especData file (especData_K.dat or especData_M.dat)
    spectrum_type : str
        Which spectrum to analyze: 'KE' (for nu) or 'PE' (for kappa)
    ratio_old : float
        OLD analysis range (default 1/3)
    ratio_new : float
        NEW analysis range (default 1/4)
    peak_ratio : float
        Range for peak finding to set truncation threshold (default 1/3)
    verbose : bool
        If True, print details and show plots
        
    Returns:
    --------
    dict with CURV values (old and new) and spectra at each time
    """
    import numpy as np
    import matplotlib.pyplot as plt
    
    direction = 'K' if 'K.dat' in filename else ('M' if 'M.dat' in filename else 'Unknown')
    
    if verbose:
        print(f"Loading {direction}-direction {spectrum_type} spectrum...")
    
    # Load the data
    with open(filename, 'r') as f:
        lines = f.readlines()
    
    # Process the data
    i = 0
    times = []
    ke_data = []
    pe_data = []
    dke_data = []
    
    while i < len(lines):
        current_set = {}
        
        while i < len(lines):
            line = lines[i].strip()
            if line:
                parts = line.split()
                label = parts[0].rstrip(':').upper()
                time_val = float(parts[1])
                data_vals = [float(x) for x in parts[2:]]
                
                current_set[label] = {'time': time_val, 'data': data_vals}
                i += 1
                
                if 'KE' in current_set and 'PE' in current_set and 'DKE' in current_set:
                    if i < len(lines):
                        next_line = lines[i].strip()
                        if next_line and next_line.split()[0].rstrip(':').upper() == 'PE2':
                            continue
                    break
            else:
                i += 1
        
        if 'KE' in current_set and 'PE' in current_set and 'DKE' in current_set:
            times.append(current_set['KE']['time'])
            ke_data.append(current_set['KE']['data'])
            pe_data.append(current_set['PE']['data'])
            dke_data.append(current_set['DKE']['data'])
    
    times = np.array(times)
    ke_data = np.array(ke_data)
    pe_data = np.array(pe_data)
    dke_data = np.array(dke_data)
    
    # Select spectrum based on type
    if spectrum_type.upper() == 'KE':
        spec_data = ke_data
    elif spectrum_type.upper() == 'PE':
        spec_data = pe_data
    else:
        raise ValueError(f"Unknown spectrum_type: {spectrum_type}. Use 'KE' or 'PE'.")
    
    # Calculate CURV for all time steps using BOTH old and new logic
    curv_old = []
    curv_new = []
    
    SPECTRAL_FLOOR = 1.0e-24
    RELATIVE_DROP_THRESHOLD = 1.0e10
    
    for t_idx, t in enumerate(times):
        E_full = np.abs(spec_data[t_idx, :])
        E_full = E_full[1:]  # Exclude zero mode
        
        N_total = len(E_full)
        
        # === OLD LOGIC: Simple 1/3 cutoff ===
        IS_old = max(1, int(N_total * ratio_old))
        E_tail_old = E_full[IS_old-1:]
        N_tail_old = len(E_tail_old)
        
        if N_tail_old < 3:
            curv_old.append(0.0)
        else:
            E_tail_safe = np.maximum(E_tail_old, 1e-50)
            LOGSP = np.log10(E_tail_safe)
            X = np.array([-1.0 + 2.0 * i / (N_tail_old - 1) for i in range(N_tail_old)])
            P2 = np.sqrt(5.0/8.0) * (3.0 * X**2 - 1.0)
            
            DX = 2.0 / (N_tail_old - 1)
            FAC = np.ones(N_tail_old)
            FAC[0] = FAC[-1] = 0.5
            
            CURV = np.sum(FAC * LOGSP * P2 * DX)
            CURV = min(10.0, max(-10.0, CURV))
            curv_old.append(CURV)
        
        # === NEW LOGIC: 1/4 cutoff with truncation ===
        # Step 1: Find peak in last peak_ratio fraction
        IS_peak = N_total - max(1, int(N_total * peak_ratio))
        PEAK_VALUE = np.max(E_full[IS_peak:])
        RELATIVE_FLOOR = PEAK_VALUE / RELATIVE_DROP_THRESHOLD
        
        # Step 2: Set analysis start
        IS_new = N_total - max(1, int(N_total * ratio_new))
        IE_new = N_total
        
        # Step 3: Truncate from end based on threshold
        # Remove modes below spectral floor (truncation errors)
        while IE_new > IS_new and (E_full[IE_new-1] < SPECTRAL_FLOOR or E_full[IE_new-1] < RELATIVE_FLOOR):
            IE_new -= 1
        
        E_tail_new = E_full[IS_new-1:IE_new]
        N_tail_new = len(E_tail_new)
        
        # If truncation removed too much data, set curvature to 0
        # (don't try to fit curvature on truncation error-dominated region)
        MIN_MODES_FOR_FIT = 10  # Need at least 10 clean modes for meaningful curvature
        
        if N_tail_new < MIN_MODES_FOR_FIT:
            curv_new.append(0.0)
        else:
            E_tail_safe = np.maximum(E_tail_new, 1e-50)
            LOGSP = np.log10(E_tail_safe)
            X = np.array([-1.0 + 2.0 * i / (N_tail_new - 1) for i in range(N_tail_new)])
            P2 = np.sqrt(5.0/8.0) * (3.0 * X**2 - 1.0)
            
            DX = 2.0 / (N_tail_new - 1)
            FAC = np.ones(N_tail_new)
            FAC[0] = FAC[-1] = 0.5
            
            CURV = np.sum(FAC * LOGSP * P2 * DX)
            CURV = min(10.0, max(-10.0, CURV))
            curv_new.append(CURV)
    
    curv_old = np.array(curv_old)
    curv_new = np.array(curv_new)
    
    if verbose:
        print(f"  {direction}-dir {spectrum_type}: {len(times)} timesteps, {spec_data.shape[1]} modes")
        print(f"  OLD CURV range: [{curv_old.min():.3f}, {curv_old.max():.3f}]")
        print(f"  NEW CURV range: [{curv_new.min():.3f}, {curv_new.max():.3f}]")
    
    return {
        'times': times,
        'curv_old': curv_old,
        'curv_new': curv_new,
        'spectra': spec_data,
        'spectrum_type': spectrum_type,
        'direction': direction,
        'n_modes': spec_data.shape[1]
    }

# Load all data quietly (no individual plots)
print("Loading simulation data...")
sim_curv_K_ke = calculate_curv_from_simulation('../../output/especData_K.dat', 'KE', verbose=True)
sim_curv_M_ke = calculate_curv_from_simulation('../../output/especData_M.dat', 'KE', verbose=True)
sim_curv_K_pe = calculate_curv_from_simulation('../../output/especData_K.dat', 'PE', verbose=True)
sim_curv_M_pe = calculate_curv_from_simulation('../../output/especData_M.dat', 'PE', verbose=True)
print("✓ Data loaded successfully\n")

In [ ]:
def comprehensive_diagnosis_hyperv():
    """
    Combined plot showing CURV (old vs new), hyperviscosity (nu), and KE spectrum evolution.
    Handles different mode counts between K and M directions.
    
    IMPORTANT: 
    - nu_T uses KE from especData_K.dat (K-direction has more modes)
    - nu_X uses KE from especData_M.dat (M-direction has fewer modes)
    - nu_R uses weighted average of both
    """
    import numpy as np
    import matplotlib.pyplot as plt
    
    print("="*80)
    print("COMPREHENSIVE DIAGNOSIS: HYPERVISCOSITY (nu) from KE")
    print("="*80)
    
    # Use pre-loaded data
    curv_times = sim_curv_K_ke['times']
    
    # OLD logic curvature
    curv_K_old = sim_curv_K_ke['curv_old']
    curv_M_old = sim_curv_M_ke['curv_old']
    curv_R_old = np.where(curv_K_old > curv_M_old,
                          (3 * curv_K_old + curv_M_old) / 4,
                          (curv_K_old + 3 * curv_M_old) / 4)
    
    # NEW logic curvature
    curv_K_new = sim_curv_K_ke['curv_new']
    curv_M_new = sim_curv_M_ke['curv_new']
    curv_R_new = np.where(curv_K_new > curv_M_new,
                          (3 * curv_K_new + curv_M_new) / 4,
                          (curv_K_new + 3 * curv_M_new) / 4)
    
    spectra_K = sim_curv_K_ke['spectra']
    spectra_M = sim_curv_M_ke['spectra']
    
    n_modes_K = sim_curv_K_ke['n_modes']
    n_modes_M = sim_curv_M_ke['n_modes']
    
    print(f"K-direction: {n_modes_K} modes")
    print(f"M-direction: {n_modes_M} modes")
    print(f"OLD logic: last 1/3 modes analyzed")
    print(f"NEW logic: last 1/4 modes with truncation")
    
    # Load hyperviscosity data
    times_hv, hyperv_data, kappa_data, nu_values = read_hyperv_data('../../output/hyperv.dat')
    
    nu_r = nu_values['hyperv']['r']
    nu_t = nu_values['hyperv']['t']
    nu_x = nu_values['hyperv']['x']
    
    # === VISUALIZATION ===
    fig = plt.figure(figsize=(18, 14))
    gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)
    
    # Row 1: CURV evolution (all directions, old vs new)
    ax1 = fig.add_subplot(gs[0, :])
    # OLD logic - dashed lines
    ax1.plot(curv_times, curv_K_old, '--', linewidth=2, ms=2, color='blue', 
             label=f'K-dir OLD (1/3, {n_modes_K} modes)', alpha=0.5)
    ax1.plot(curv_times, curv_M_old, '--', linewidth=2, ms=2, color='green', 
             label=f'M-dir OLD (1/3, {n_modes_M} modes)', alpha=0.5)
    ax1.plot(curv_times, curv_R_old, '--', linewidth=2, ms=2, color='red', 
             label='Avg OLD (1/3)', alpha=0.5)
    # NEW logic - solid lines
    ax1.plot(curv_times, curv_K_new, '-', linewidth=2.5, ms=3, color='blue', 
             label=f'K-dir NEW (1/4)', alpha=0.8)
    ax1.plot(curv_times, curv_M_new, '-', linewidth=2.5, ms=3, color='green', 
             label=f'M-dir NEW (1/4)', alpha=0.8)
    ax1.plot(curv_times, curv_R_new, '-', linewidth=2.5, ms=3, color='red', 
             label='Avg NEW (1/4)', alpha=0.8)
    ax1.axhline(0, color='gray', linestyle='-', alpha=0.5, linewidth=1.5, label='Target≈0')
    ax1.axhline(-0.3, color='orange', linestyle='--', alpha=0.5, label='Old target=-0.3')
    ax1.axhline(-2.0, color='purple', linestyle=':', alpha=0.5, label='Over-damp<-2')
    ax1.set_xlabel('Time', fontsize=11)
    ax1.set_ylabel('CURV (from KE)', fontsize=11)
    ax1.set_title('KE Spectral Curvature: OLD (dashed) vs NEW (solid) Logic', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=8, ncol=3, loc='best')
    ax1.grid(True, alpha=0.3)
    
    # Row 2: Hyperviscosity evolution
    ax2 = fig.add_subplot(gs[1, :])
    ax2.plot(times_hv, nu_r, 'o-', linewidth=2, ms=3, label='nu_R (weighted)', color='red', alpha=0.8)
    ax2.plot(times_hv, nu_t, 's-', linewidth=2, ms=3, label='nu_T (K-dir)', color='blue', alpha=0.8)
    ax2.plot(times_hv, nu_x, '^-', linewidth=2, ms=3, label='nu_X (M-dir)', color='green', alpha=0.8)
    ax2.set_xlabel('Time', fontsize=11)
    ax2.set_ylabel('Hyperviscosity nu', fontsize=11)
    ax2.set_title('Hyperviscosity Evolution (Actual)', fontsize=12, fontweight='bold')
    ax2.set_yscale('log')
    ax2.legend(fontsize=9, ncol=3)
    ax2.grid(True, alpha=0.3)
    
    # Row 3: KE Spectrum snapshots (K/M/comparison)
    dt_target = 1.0
    selected_times = []
    selected_idx = []
    for i, t in enumerate(curv_times):
        if len(selected_times) == 0 or t >= selected_times[-1] + dt_target:
            selected_times.append(t)
            selected_idx.append(i)
    
    n_selected = len(selected_idx)
    colors_time = plt.cm.coolwarm(np.linspace(0, 1, n_selected))
    
    # K-direction KE
    ax3a = fig.add_subplot(gs[2, 0])
    for i, (idx, color) in enumerate(zip(selected_idx, colors_time)):
        E = np.abs(spectra_K[idx, 1:])
        modes = np.arange(1, len(E)+1)
        alpha_val = 0.4 if i < n_selected-1 else 0.9
        lw_val = 1.0 if i < n_selected-1 else 2.5
        ms_val = 2 if i < n_selected-1 else 4
        ax3a.loglog(modes, E, 'o-', color=color, lw=lw_val, ms=ms_val, alpha=alpha_val, 
                   label=f't={curv_times[idx]:.1f}' if i == 0 or i == n_selected-1 else '')
        
        tail_idx = int(0.85 * len(modes))
        ax3a.text(modes[tail_idx], E[tail_idx], f' {curv_times[idx]:.1f}', 
                 fontsize=7, color='black', alpha=1.0, ha='left', va='center')
    ax3a.axvline(x=n_modes_K*2/3, color='red', linestyle='--', alpha=0.5,
                 label=f'2/3 cut-off')
    ax3a.axvline(x=n_modes_K*3/4, color='orange', linestyle=':', alpha=0.5,
                 label=f'3/4 (NEW start)')
    ax3a.set_xlabel('Mode Number', fontsize=10)
    ax3a.set_ylabel('KE Energy', fontsize=10)
    ax3a.set_title(f'KE K-dir (nu_T, {n_modes_K} modes)', fontsize=11, fontweight='bold')
    ax3a.legend(fontsize=8, loc='best')
    ax3a.grid(True, alpha=0.3, which='both')
    
    # M-direction KE
    ax3b = fig.add_subplot(gs[2, 1])
    for i, (idx, color) in enumerate(zip(selected_idx, colors_time)):
        E = np.abs(spectra_M[idx, 1:])
        modes = np.arange(1, len(E)+1)
        alpha_val = 0.4 if i < n_selected-1 else 0.9
        lw_val = 1.0 if i < n_selected-1 else 2.5
        ms_val = 2 if i < n_selected-1 else 4
        ax3b.loglog(modes, E, 'o-', color=color, lw=lw_val, ms=ms_val, alpha=alpha_val,
                   label=f't={curv_times[idx]:.1f}' if i == 0 or i == n_selected-1 else '')
        
        tail_idx = int(0.85 * len(modes))
        ax3b.text(modes[tail_idx], E[tail_idx], f' {curv_times[idx]:.1f}', 
                 fontsize=7, color='black', alpha=1.0, ha='left', va='center')
    ax3b.axvline(x=n_modes_M*2/3, color='red', linestyle='--', alpha=0.5,
                 label=f'2/3 cut-off')
    ax3b.axvline(x=n_modes_M*3/4, color='orange', linestyle=':', alpha=0.5,
                 label=f'3/4 (NEW start)')
    ax3b.set_xlabel('Mode Number', fontsize=10)
    ax3b.set_ylabel('KE Energy', fontsize=10)
    ax3b.set_title(f'KE M-dir (nu_X, {n_modes_M} modes)', fontsize=11, fontweight='bold')
    ax3b.legend(fontsize=8, loc='best')
    ax3b.grid(True, alpha=0.3, which='both')
    
    # Final time KE comparison with k^(-5/3) reference
    ax3c = fig.add_subplot(gs[2, 2])
    E_K = np.abs(spectra_K[-1, 1:])
    E_M = np.abs(spectra_M[-1, 1:])
    modes_K = np.arange(1, len(E_K)+1)
    modes_M = np.arange(1, len(E_M)+1)
    
    ax3c.loglog(modes_K, E_K, 'o-', color='blue', lw=2, ms=4, alpha=0.7, label=f'K ({n_modes_K})')
    ax3c.loglog(modes_M, E_M, 's-', color='green', lw=2, ms=4, alpha=0.7, label=f'M ({n_modes_M})')
    
    k_max = modes_K[-1]
    E_max = E_K[-1]
    k_ref = np.logspace(np.log10(k_max/10), np.log10(k_max), 50)
    E_ref = E_max * (k_ref / k_max)**(-5.0/3.0)
    ax3c.loglog(k_ref, E_ref, 'k--', lw=2, alpha=0.8, label=r'$k^{-5/3}$')
    ax3c.axvline(x=int(n_modes_K*2/3), color='red', linestyle='--', alpha=0.5,
                 label=f'2/3 cut-off')
    ax3c.axvline(x=int(n_modes_K*3/4), color='orange', linestyle=':', alpha=0.5,
                 label=f'3/4 (NEW)')
    
    ax3c.set_xlabel('Mode Number', fontsize=10)
    ax3c.set_ylabel('KE Energy', fontsize=10)
    ax3c.set_title(f'KE Final (t={curv_times[-1]:.1f})', fontsize=11, fontweight='bold')
    ax3c.legend(fontsize=8, loc='best')
    ax3c.grid(True, alpha=0.3, which='both')
    
    plt.suptitle('Comprehensive Diagnosis: Hyperviscosity (nu) - OLD vs NEW Curvature Logic', 
                fontsize=15, fontweight='bold', y=0.995)
    plt.show()
    
    print("\nCURVATURE COMPARISON:")
    print(f"OLD logic final: K={curv_K_old[-1]:.3f}, M={curv_M_old[-1]:.3f}, R={curv_R_old[-1]:.3f}")
    print(f"NEW logic final: K={curv_K_new[-1]:.3f}, M={curv_M_new[-1]:.3f}, R={curv_R_new[-1]:.3f}")
    print(f"\nNEW target should be ≈ 0 to -0.1 (straight dissipative tail)")
    print(f"Current implementation uses target = -0.3 (too negative for 1/4 range)")
    
    return {
        'times': curv_times,
        'curv_K_old': curv_K_old, 'curv_M_old': curv_M_old, 'curv_R_old': curv_R_old,
        'curv_K_new': curv_K_new, 'curv_M_new': curv_M_new, 'curv_R_new': curv_R_new,
        'nu_times': times_hv,
        'nu_r': nu_r, 'nu_t': nu_t, 'nu_x': nu_x,
        'spectra_K': spectra_K, 'spectra_M': spectra_M,
        'n_modes_K': n_modes_K, 'n_modes_M': n_modes_M
    }

In [ ]:
def comprehensive_diagnosis_hyperdiff():
    """
    Combined plot showing CURV (old vs new), hyperdiffusivity (kappa), and PE spectrum evolution.
    Handles different mode counts between K and M directions.
    
    IMPORTANT: 
    - kappa_T uses PE from especData_K.dat (K-direction has more modes)
    - kappa_X uses PE from especData_M.dat (M-direction has fewer modes)
    - kappa_R uses weighted average of both
    """
    import numpy as np
    import matplotlib.pyplot as plt
    
    print("="*80)
    print("COMPREHENSIVE DIAGNOSIS: HYPERDIFFUSIVITY (kappa) from PE")
    print("="*80)
    
    # Use pre-loaded data
    curv_times = sim_curv_K_pe['times']
    
    # OLD logic curvature
    curv_K_old = sim_curv_K_pe['curv_old']
    curv_M_old = sim_curv_M_pe['curv_old']
    curv_R_old = np.where(curv_K_old > curv_M_old,
                          (3 * curv_K_old + curv_M_old) / 4,
                          (curv_K_old + 3 * curv_M_old) / 4)
    
    # NEW logic curvature
    curv_K_new = sim_curv_K_pe['curv_new']
    curv_M_new = sim_curv_M_pe['curv_new']
    curv_R_new = np.where(curv_K_new > curv_M_new,
                          (3 * curv_K_new + curv_M_new) / 4,
                          (curv_K_new + 3 * curv_M_new) / 4)
    
    spectra_K = sim_curv_K_pe['spectra']
    spectra_M = sim_curv_M_pe['spectra']
    
    n_modes_K = sim_curv_K_pe['n_modes']
    n_modes_M = sim_curv_M_pe['n_modes']
    
    print(f"K-direction: {n_modes_K} modes")
    print(f"M-direction: {n_modes_M} modes")
    print(f"OLD logic: last 1/3 modes analyzed")
    print(f"NEW logic: last 1/4 modes with truncation")
    
    # Load hyperdiffusivity data
    times_hv, hyperv_data, kappa_data, nu_values = read_hyperv_data('../../output/hyperv.dat')
    
    kappa_r = nu_values['kappa']['r']
    kappa_t = nu_values['kappa']['t']
    kappa_x = nu_values['kappa']['x']
    
    # === VISUALIZATION ===
    fig = plt.figure(figsize=(18, 14))
    gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)
    
    # Row 1: CURV evolution (all directions, old vs new)
    ax1 = fig.add_subplot(gs[0, :])
    # OLD logic - dashed lines
    ax1.plot(curv_times, curv_K_old, '--', linewidth=2, ms=2, color='blue', 
             label=f'K-dir OLD (1/3, {n_modes_K} modes)', alpha=0.5)
    ax1.plot(curv_times, curv_M_old, '--', linewidth=2, ms=2, color='green', 
             label=f'M-dir OLD (1/3, {n_modes_M} modes)', alpha=0.5)
    ax1.plot(curv_times, curv_R_old, '--', linewidth=2, ms=2, color='red', 
             label='Avg OLD (1/3)', alpha=0.5)
    # NEW logic - solid lines
    ax1.plot(curv_times, curv_K_new, '-', linewidth=2.5, ms=3, color='blue', 
             label=f'K-dir NEW (1/4)', alpha=0.8)
    ax1.plot(curv_times, curv_M_new, '-', linewidth=2.5, ms=3, color='green', 
             label=f'M-dir NEW (1/4)', alpha=0.8)
    ax1.plot(curv_times, curv_R_new, '-', linewidth=2.5, ms=3, color='red', 
             label='Avg NEW (1/4)', alpha=0.8)
    ax1.axhline(0, color='gray', linestyle='-', alpha=0.5, linewidth=1.5, label='Target≈0')
    ax1.axhline(-0.3, color='orange', linestyle='--', alpha=0.5, label='Old target=-0.3')
    ax1.axhline(-2.0, color='purple', linestyle=':', alpha=0.5, label='Over-damp<-2')
    ax1.set_xlabel('Time', fontsize=11)
    ax1.set_ylabel('CURV (from PE)', fontsize=11)
    ax1.set_title('PE Spectral Curvature: OLD (dashed) vs NEW (solid) Logic', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=8, ncol=3, loc='best')
    ax1.grid(True, alpha=0.3)
    
    # Row 2: Hyperdiffusivity evolution
    ax2 = fig.add_subplot(gs[1, :])
    ax2.plot(times_hv, kappa_r, 'o-', linewidth=2, ms=3, label='kappa_R (weighted)', color='red', alpha=0.8)
    ax2.plot(times_hv, kappa_t, 's-', linewidth=2, ms=3, label='kappa_T (K-dir)', color='blue', alpha=0.8)
    ax2.plot(times_hv, kappa_x, '^-', linewidth=2, ms=3, label='kappa_X (M-dir)', color='green', alpha=0.8)
    ax2.set_xlabel('Time', fontsize=11)
    ax2.set_ylabel('Hyperdiffusivity kappa', fontsize=11)
    ax2.set_title('Hyperdiffusivity Evolution (Actual)', fontsize=12, fontweight='bold')
    ax2.set_yscale('log')
    ax2.legend(fontsize=9, ncol=3)
    ax2.grid(True, alpha=0.3)
    
    # Row 3: PE Spectrum snapshots (K/M/comparison)
    dt_target = 1.0
    selected_times = []
    selected_idx = []
    for i, t in enumerate(curv_times):
        if len(selected_times) == 0 or t >= selected_times[-1] + dt_target:
            selected_times.append(t)
            selected_idx.append(i)
    
    n_selected = len(selected_idx)
    colors_time = plt.cm.coolwarm(np.linspace(0, 1, n_selected))
    
    # K-direction PE
    ax3a = fig.add_subplot(gs[2, 0])
    for i, (idx, color) in enumerate(zip(selected_idx, colors_time)):
        E = np.abs(spectra_K[idx, 1:])
        modes = np.arange(1, len(E)+1)
        alpha_val = 0.4 if i < n_selected-1 else 0.9
        lw_val = 1.0 if i < n_selected-1 else 2.5
        ms_val = 2 if i < n_selected-1 else 4
        ax3a.loglog(modes, E, 'o-', color=color, lw=lw_val, ms=ms_val, alpha=alpha_val, 
                   label=f't={curv_times[idx]:.1f}' if i == 0 or i == n_selected-1 else '')
        
        tail_idx = int(0.85 * len(modes))
        ax3a.text(modes[tail_idx], E[tail_idx], f' {curv_times[idx]:.1f}', 
                 fontsize=7, color='black', alpha=1.0, ha='left', va='center')
    ax3a.axvline(x=n_modes_K*2/3, color='red', linestyle='--', alpha=0.5,
                 label=f'2/3 cut-off')
    ax3a.axvline(x=n_modes_K*3/4, color='orange', linestyle=':', alpha=0.5,
                 label=f'3/4 (NEW start)')
    ax3a.set_xlabel('Mode Number', fontsize=10)
    ax3a.set_ylabel('PE Energy', fontsize=10)
    ax3a.set_title(f'PE K-dir (kappa_T, {n_modes_K} modes)', fontsize=11, fontweight='bold')
    ax3a.legend(fontsize=8, loc='best')
    ax3a.grid(True, alpha=0.3, which='both')
    
    # M-direction PE
    ax3b = fig.add_subplot(gs[2, 1])
    for i, (idx, color) in enumerate(zip(selected_idx, colors_time)):
        E = np.abs(spectra_M[idx, 1:])
        modes = np.arange(1, len(E)+1)
        alpha_val = 0.4 if i < n_selected-1 else 0.9
        lw_val = 1.0 if i < n_selected-1 else 2.5
        ms_val = 2 if i < n_selected-1 else 4
        ax3b.loglog(modes, E, 'o-', color=color, lw=lw_val, ms=ms_val, alpha=alpha_val,
                   label=f't={curv_times[idx]:.1f}' if i == 0 or i == n_selected-1 else '')
        
        tail_idx = int(0.85 * len(modes))
        ax3b.text(modes[tail_idx], E[tail_idx], f' {curv_times[idx]:.1f}', 
                 fontsize=7, color='black', alpha=1.0, ha='left', va='center')
    ax3b.axvline(x=n_modes_M*2/3, color='red', linestyle='--', alpha=0.5,
                 label=f'2/3 cut-off')
    ax3b.axvline(x=n_modes_M*3/4, color='orange', linestyle=':', alpha=0.5,
                 label=f'3/4 (NEW start)')
    ax3b.set_xlabel('Mode Number', fontsize=10)
    ax3b.set_ylabel('PE Energy', fontsize=10)
    ax3b.set_title(f'PE M-dir (kappa_X, {n_modes_M} modes)', fontsize=11, fontweight='bold')
    ax3b.legend(fontsize=8, loc='best')
    ax3b.grid(True, alpha=0.3, which='both')
    
    # Final time PE comparison with k^(-5/3) reference
    ax3c = fig.add_subplot(gs[2, 2])
    E_K = np.abs(spectra_K[-1, 1:])
    E_M = np.abs(spectra_M[-1, 1:])
    modes_K = np.arange(1, len(E_K)+1)
    modes_M = np.arange(1, len(E_M)+1)
    
    ax3c.loglog(modes_K, E_K, 'o-', color='blue', lw=2, ms=4, alpha=0.7, label=f'K ({n_modes_K})')
    ax3c.loglog(modes_M, E_M, 's-', color='green', lw=2, ms=4, alpha=0.7, label=f'M ({n_modes_M})')
    
    k_max = modes_K[-1]
    E_max = E_K[-1]
    k_ref = np.logspace(np.log10(k_max/10), np.log10(k_max), 50)
    E_ref = E_max * (k_ref / k_max)**(-5.0/3.0)
    ax3c.loglog(k_ref, E_ref, 'k--', lw=2, alpha=0.8, label=r'$k^{-5/3}$')
    ax3c.axvline(x=int(n_modes_K*2/3), color='red', linestyle='--', alpha=0.5,
                 label=f'2/3 cut-off')
    ax3c.axvline(x=int(n_modes_K*3/4), color='orange', linestyle=':', alpha=0.5,
                 label=f'3/4 (NEW)')
    
    ax3c.set_xlabel('Mode Number', fontsize=10)
    ax3c.set_ylabel('PE Energy', fontsize=10)
    ax3c.set_title(f'PE Final (t={curv_times[-1]:.1f})', fontsize=11, fontweight='bold')
    ax3c.legend(fontsize=8, loc='best')
    ax3c.grid(True, alpha=0.3, which='both')
    
    plt.suptitle('Comprehensive Diagnosis: Hyperdiffusivity (kappa) - OLD vs NEW Curvature Logic', 
                fontsize=15, fontweight='bold', y=0.995)
    plt.show()
    
    print("\nCURVATURE COMPARISON:")
    print(f"OLD logic final: K={curv_K_old[-1]:.3f}, M={curv_M_old[-1]:.3f}, R={curv_R_old[-1]:.3f}")
    print(f"NEW logic final: K={curv_K_new[-1]:.3f}, M={curv_M_new[-1]:.3f}, R={curv_R_new[-1]:.3f}")
    print(f"\nNEW target should be ≈ 0 to -0.1 (straight dissipative tail)")
    print(f"Current implementation uses target = -0.3 (too negative for 1/4 range)")
    
    return {
        'times': curv_times,
        'curv_K_old': curv_K_old, 'curv_M_old': curv_M_old, 'curv_R_old': curv_R_old,
        'curv_K_new': curv_K_new, 'curv_M_new': curv_M_new, 'curv_R_new': curv_R_new,
        'kappa_times': times_hv,
        'kappa_r': kappa_r, 'kappa_t': kappa_t, 'kappa_x': kappa_x,
        'spectra_K': spectra_K, 'spectra_M': spectra_M,
        'n_modes_K': n_modes_K, 'n_modes_M': n_modes_M
    }

### Results

In [ ]:
hyperv_data = comprehensive_diagnosis_hyperv()
hyperdiff_data = comprehensive_diagnosis_hyperdiff()